# Detección de casco y chaleco en obras de construcción mediante YOLO26

Comparación de **tres detectores** bajo protocolo idéntico:

| Papel | Modelo |
|---|---|
| Línea base simple | `yolov8n` |
| Comparador directo | `yolo11n` |
| Modelo propuesto | `yolo26n` |

Mismos datos, mismos hiperparámetros, misma semilla. Es lo que hace válida la comparación.

**Antes de la corrida larga**, deja `PRUEBA_RAPIDA = True` para validar que todo el flujo
funciona en unos minutos. Solo entonces pásalo a `False`.


## 1 · Entorno


In [ ]:
import subprocess, sys, platform
print('Python :', platform.python_version())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'SIN GPU')


In [ ]:
%pip install -q -U ultralytics
import torch, ultralytics
print('torch       :', torch.__version__)
print('ultralytics :', ultralytics.__version__)
print('CUDA        :', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Activa la GPU en Settings > Accelerator'


## 2 · Configuración

Todo lo ajustable vive en esta celda.


In [ ]:
from pathlib import Path

PRUEBA_RAPIDA = True      # True = 2 épocas, subconjunto. False = corrida real

MODELOS = {
    'yolov8n': 'Línea base simple',
    'yolo11n': 'Comparador directo',
    'yolo26n': 'Modelo propuesto',
}

# Protocolo IDÉNTICO para los tres. No cambiar entre modelos.
PROTOCOLO = dict(
    epochs   = 2 if PRUEBA_RAPIDA else 50,
    imgsz    = 640,
    batch    = 32,
    optimizer= 'auto',
    seed     = 42,
    patience = 15,
    device   = 0,
    workers  = 4,
    val      = True,
    plots    = True,
    exist_ok = True,
)

TRABAJO = Path('/kaggle/working')
INPUT   = Path('/kaggle/input')

# --- localizar el dataset: Kaggle lo monta anidado, p.ej.
#     /kaggle/input/datasets/<usuario>/<slug>/   ---
def localizar_dataset(max_prof=5):
    """Busca en anchura la carpeta que contenga data.yaml y train/images."""
    if not INPUT.exists():
        return None
    cola = [(INPUT, 0)]
    while cola:
        d, prof = cola.pop(0)
        if (d/'data.yaml').exists() and (d/'train'/'images').is_dir():
            return d
        if prof < max_prof:
            try:
                cola += [(s, prof+1) for s in sorted(d.iterdir()) if s.is_dir()]
            except (PermissionError, OSError):
                pass
    return None


def mostrar_arbol(d, prof=0, max_prof=3):
    if prof > max_prof:
        return
    try:
        hijos = sorted(d.iterdir())
    except (PermissionError, OSError):
        return
    for s in hijos[:10]:
        print('   '*(prof+1) + s.name + ('/' if s.is_dir() else ''))
        if s.is_dir():
            mostrar_arbol(s, prof+1, max_prof)
    if len(hijos) > 10:
        print('   '*(prof+1) + f'... y {len(hijos)-10} más')

RAIZ = localizar_dataset()

if RAIZ is None:
    print('No se encontró el dataset. Árbol de /kaggle/input:\n')
    if INPUT.exists():
        mostrar_arbol(INPUT)
    else:
        print('    /kaggle/input no existe: no hay ningún input montado.')
    raise SystemExit(
        '\nSOLUCIÓN: en el panel derecho, Input > Add Input, busca '
        '"epp-casco-chaleco-yolo26" y añádelo. '
        'Si ya aparece, reinicia con Run > Restart & clear cell outputs.'
    )

print('Dataset encontrado en:', RAIZ)
print('Protocolo :', PROTOCOLO)


## 3 · Datos

`/kaggle/input` es de solo lectura, así que el `data.yaml` se escribe en `/kaggle/working`.


In [ ]:
import yaml, random, shutil

def contar(split):
    d = RAIZ/split/'images'
    return len(list(d.iterdir())) if d.is_dir() else 0

for s in ('train','val','test'):
    print(f'  {s:<6} {contar(s):>6} imágenes')

origen = RAIZ

if PRUEBA_RAPIDA:
    # Subconjunto diminuto para validar el flujo sin gastar GPU.
    origen = TRABAJO/'muestra'
    if origen.exists(): shutil.rmtree(origen)
    random.seed(42)
    for s, n in (('train',200), ('val',60), ('test',60)):
        (origen/s/'images').mkdir(parents=True); (origen/s/'labels').mkdir(parents=True)
        imgs = sorted((RAIZ/s/'images').iterdir())
        for img in random.sample(imgs, min(n, len(imgs))):
            lbl = RAIZ/s/'labels'/(img.stem + '.txt')
            if lbl.exists():
                shutil.copy2(img, origen/s/'images'/img.name)
                shutil.copy2(lbl, origen/s/'labels'/lbl.name)
    print('\nMuestra creada en', origen)

DATA_YAML = TRABAJO/'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path' : str(origen),
    'train': 'train/images',
    'val'  : 'val/images',
    'test' : 'test/images',
    'nc'   : 2,
    'names': {0: 'casco', 1: 'chaleco'},
}, sort_keys=False, allow_unicode=True))
print(DATA_YAML.read_text())


## 4 · Entrenamiento

Los tres modelos, en secuencia, con el mismo protocolo.


In [ ]:
import time, gc, torch
from ultralytics import YOLO

PROYECTO = str(TRABAJO/'runs')
tiempos = {}

for nombre, papel in MODELOS.items():
    print(f'\n{"="*70}\n{nombre}  ·  {papel}\n{"="*70}')
    t0 = time.time()
    modelo = YOLO(f'{nombre}.pt')          # pesos COCO -> transfer learning
    modelo.train(data=str(DATA_YAML), project=PROYECTO, name=nombre, **PROTOCOLO)
    tiempos[nombre] = time.time() - t0
    print(f'  tiempo: {tiempos[nombre]/60:.1f} min')
    del modelo; gc.collect(); torch.cuda.empty_cache()

print('\nEntrenamiento completo.')
for n, t in tiempos.items():
    print(f'  {n:<10} {t/60:>7.1f} min')


## 5 · Evaluación sobre el conjunto de prueba

Ninguno de los modelos vio estas imágenes durante el entrenamiento.


In [ ]:
import pandas as pd

filas = []
for nombre, papel in MODELOS.items():
    pesos = Path(PROYECTO)/nombre/'weights'/'best.pt'
    m = YOLO(str(pesos))
    r = m.val(data=str(DATA_YAML), split='test', imgsz=PROTOCOLO['imgsz'],
              batch=PROTOCOLO['batch'], device=0, plots=True,
              project=PROYECTO, name=f'{nombre}_test', exist_ok=True)
    b = r.box
    filas.append({
        'modelo'      : nombre,
        'papel'       : papel,
        'mAP50'       : round(float(b.map50), 4),
        'mAP50-95'    : round(float(b.map), 4),
        'precision'   : round(float(b.mp), 4),
        'recall'      : round(float(b.mr), 4),
        'mAP50_casco' : round(float(b.ap50[0]), 4),
        'mAP50_chaleco': round(float(b.ap50[1]), 4),
        'parametros'  : sum(p.numel() for p in m.model.parameters()),
        'min_entren'  : round(tiempos.get(nombre, 0)/60, 1),
    })
    del m; gc.collect(); torch.cuda.empty_cache()

tabla = pd.DataFrame(filas)
tabla.to_csv(TRABAJO/'comparativa_modelos.csv', index=False)
tabla


### Tabla para el artículo


In [ ]:
cols = ['modelo','mAP50','mAP50-95','precision','recall','mAP50_casco','mAP50_chaleco']
print(tabla[cols].to_markdown(index=False))

mejor = tabla.loc[tabla['mAP50-95'].idxmax()]
base  = tabla.loc[tabla['modelo'] == 'yolov8n'].iloc[0]
print(f"\nMejor modelo: {mejor['modelo']} con mAP50-95 = {mejor['mAP50-95']:.4f}")
print(f"Mejora sobre la línea base: {mejor['mAP50-95'] - base['mAP50-95']:+.4f} "
      f"({100*(mejor['mAP50-95']/base['mAP50-95'] - 1):+.1f} %)")
print(f"\nBrecha casco vs chaleco en {mejor['modelo']}: "
      f"{mejor['mAP50_casco'] - mejor['mAP50_chaleco']:+.4f}")


## 6 · Figuras

Ultralytics las genera durante el entrenamiento y la validación.


In [ ]:
from IPython.display import Image, display

for nombre in MODELOS:
    for fig in ('confusion_matrix_normalized.png', 'results.png', 'val_batch0_pred.jpg'):
        for sub in (nombre, f'{nombre}_test'):
            p = Path(PROYECTO)/sub/fig
            if p.exists():
                print(f'\n--- {sub} / {fig}')
                display(Image(filename=str(p), width=680))
                break


## 7 · Exportar resultados

Los pesos del mejor modelo van al VPS para servir la demo.


In [ ]:
import shutil, json

SALIDA = TRABAJO/'entregables'
SALIDA.mkdir(exist_ok=True)

for nombre in MODELOS:
    w = Path(PROYECTO)/nombre/'weights'/'best.pt'
    if w.exists(): shutil.copy2(w, SALIDA/f'{nombre}_best.pt')
    r = Path(PROYECTO)/nombre/'results.csv'
    if r.exists(): shutil.copy2(r, SALIDA/f'{nombre}_results.csv')

tabla.to_csv(SALIDA/'comparativa_modelos.csv', index=False)
(SALIDA/'protocolo.json').write_text(json.dumps(
    {'protocolo': PROTOCOLO, 'modelos': MODELOS,
     'prueba_rapida': PRUEBA_RAPIDA, 'tiempos_min': {k: round(v/60,1) for k,v in tiempos.items()}},
    indent=2, ensure_ascii=False))

for f in sorted(SALIDA.iterdir()):
    print(f'  {f.stat().st_size/1048576:8.1f} MB  {f.name}')
print('\nDescarga la carpeta entregables/ desde el panel Output de Kaggle.')


## Siguiente paso

Con `yolo26n_best.pt` descargado, desplegarlo en el VPS:

```bash
scp yolo26n_best.pt root@213.136.69.236:/opt/epp-yolo26/pesos/best.pt
ssh root@213.136.69.236 "cd /opt/epp-yolo26 && docker compose restart"
ssh root@213.136.69.236 "curl -s localhost:8090/salud"
```

Debe responder `"pesos_presentes": true`.
